# Update dfs w/ new variables, column names etc. 

- instead of re-running dfmaker for now
- but this should be merged into makedf at some point

In [1]:
import pandas as pd
import numpy as np
from os import path

import sys
# sys.path.append('../../../')
sys.path.append('/exp/sbnd/app/users/munjung/xsec/cafpyana_2026Jan17/cafpyana') # absolute path for running on EAF
from analysis_village.numucc_1p0pi.utils import *
from analysis_village.numucc_1p0pi.categories import *
from pyanalib.variable_calculator import *

# turn off PerformanceWarning 
# triggered by mismatched column levels
import warnings
warnings.filterwarnings("ignore", category=pd.errors.PerformanceWarning)
# also turn off NaturalNameWarning (from PyTables)
import tables
warnings.filterwarnings("ignore", category=tables.NaturalNameWarning)

In [2]:
from makedf.g4syst import g4_systematics
from makedf.mcstat import get_MCstat_unc

In [3]:
def add_opening_angle(df, truth=False, nu=False):
    if nu:
        opening_angle = (
            df.mc.mu.dir[["x", "y", "z"]].values * df.mc.p.dir[["x", "y", "z"]].values
        ).sum(axis=1)

        df["theta_mu_p"] = np.arccos(opening_angle) * 180. / np.pi

    elif truth:
        opening_angle = (
            df.mc.mu.dir[["x", "y", "z"]].values * df.mc.p.dir[["x", "y", "z"]].values
        ).sum(axis=1)

        df["mc_theta_mu_p"] = np.arccos(opening_angle) * 180. / np.pi


    else:
        opening_angle = (
            df.mu.pfp.trk.dir[["x", "y", "z"]].values * df.p.pfp.trk.dir[["x", "y", "z"]].values
        ).sum(axis=1)

        df["theta_mu_p"] = np.arccos(opening_angle) * 180. / np.pi
    return df

In [4]:
def add_track_direction(df):
    print("adding track direction")
    df.loc[:, ("mu", "pfp","trk","truth","p","dir","x")] = df.mu.pfp.trk.truth.p.genp.x / df.mu.pfp.trk.truth.p.totp
    df.loc[:, ("mu", "pfp","trk","truth","p","dir","y")] = df.mu.pfp.trk.truth.p.genp.y / df.mu.pfp.trk.truth.p.totp
    df.loc[:, ("mu", "pfp","trk","truth","p","dir","z")] = df.mu.pfp.trk.truth.p.genp.z / df.mu.pfp.trk.truth.p.totp
    df.loc[:, ("p", "pfp","trk","truth","p","dir","x")] = df.p.pfp.trk.truth.p.genp.x / df.p.pfp.trk.truth.p.totp
    df.loc[:, ("p", "pfp","trk","truth","p","dir","y")] = df.p.pfp.trk.truth.p.genp.y / df.p.pfp.trk.truth.p.totp
    df.loc[:, ("p", "pfp","trk","truth","p","dir","z")] = df.p.pfp.trk.truth.p.genp.z / df.p.pfp.trk.truth.p.totp
    return df

tki_var_names = ["del_alpha", "del_phi", "del_Tp", "del_p", "del_Tp_x", "del_Tp_y"]

## Modify just evtdf

In [23]:
file_dir = "/exp/sbnd/data/users/munjung/xsec/2025Spring_v10_06_00_09"
# file_dir="/pnfs/sbnd/scratch/users/munjung/xsec/2025Spring_v10_06_00_09"
sub_dir="MC"
sample_dir="BNB_cosmics"

In [ ]:
# modify evtdf

# for tag in generate_tags("bl"):
for tag in ["GiBUU-sel_all"]:
    print("file tag:", tag)
    filename = path.join(file_dir, sub_dir, sample_dir, f"{tag}.df")
    # filename = path.join(file_dir, sub_dir, sample_dir, f"{tag}_geniewgts_CCQE.df")
    # filename = path.join(file_dir, sub_dir, sample_dir, f"{tag}_sel_mup-geniewgts.df")
    this_split_df = pd.read_hdf(filename, key="split")
    this_n_split = this_split_df.n_split.iloc[0]

    with pd.HDFStore(filename, mode='r') as store:
        keys = store.keys() 
        pass
        # print("Keys:", keys)

    for i in range(this_n_split):
        evt_key = f"evt_{i}"
        hdr_key = f"hdr_{i}"
        this_df = pd.read_hdf(filename, key=evt_key)
        this_hdr_df = pd.read_hdf(filename, key=hdr_key)

        # ====== add stuff to the df ======
        try:
            # # MC stat
            # this_df, _ = get_MCstat_unc(this_df, this_hdr_df, n_universes=1000)

            # evetnt categories
            # this_df.loc[:,'topo_categ'] = get_topo_category(this_df)
            # this_df.loc[:,'genie_categ'] = get_genie_category(this_df)

            # add track direction
            # this_df = add_track_direction(this_df)

            # opening angle
            this_df = add_opening_angle(this_df)
            this_df = add_opening_angle(this_df, truth=True)


            # # add tki
            # slc_mudf = this_df.mu.pfp.trk.truth.p
            # slc_pdf = this_df.p.pfp.trk.truth.p
            # slc_P_mu_col = pad_column_name(("totp",), slc_mudf)
            # slc_P_p_col = pad_column_name(("totp",), slc_pdf)
            # tki_reco = get_cc1p0pi_tki(slc_mudf, slc_pdf, slc_P_mu_col, slc_P_p_col)
            # for var_name in tki_var_names:
            #     this_df = multicol_add(this_df, tki_reco[var_name].rename("mc_" + var_name))

            # # slim G4 syst
            # g4_cols = [col for col in this_df.mc.columns if col[0] in g4_systematics]
            # universes = sorted({col[1] for col in g4_cols})
            # G4_wgts = pd.DataFrame(index=this_df.mc.index)

            # for univ in universes:
            #     univ_cols = [(syst, univ, "", "", "", "") for syst in g4_systematics if (syst, univ) in this_df.mc.columns]
            #     if len(univ_cols) == len(g4_systematics):  # only use if all present for this universe
            #         G4_wgts[univ] = this_df.mc.loc[:, univ_cols].prod(axis=1)
            #     else:
            #         # fallback: fill with 1. (could also raise or warn)
            #         G4_wgts[univ] = 1.0
            # for univ in universes:
            #     this_df[("mc", "G4", univ, "", "", "", "")] = G4_wgts[univ]

        # except:
        #     # raise ValueError(f"Error for {evt_key}")
        #     print(f"Error for {evt_key}")
        #     continue
        # ==================================

        # overwrite df with the same structure as the original df
        with pd.HDFStore(filename, mode='r+') as store:
            store.put(evt_key, this_df)

file tag: GiBUU-sel_all


AttributeError: 'DataFrame' object has no attribute 'mu'

In [ ]:
# # check
# with pd.HDFStore(filename, mode='r') as store:
#     keys = store.keys()       # list of all keys in the file
#     print("Keys:", keys)

# type_key = "evt"
# for i in range(this_n_split):
#     this_key = f"{type_key}_{i}"
#     this_df = pd.read_hdf(filename, key=this_key)
#     print(this_df.MCstat)

## Modify nudf

In [ ]:
file_dir = "/exp/sbnd/data/users/munjung/xsec/2025Spring_v10_06_00_09"
sub_dir="MC"
sample_dir="BNB_cosmics/genie_wgts-Other"

for tag in generate_tags("bl"):
    tag += ""
    print("file tag:", tag)
    filename = path.join(file_dir, sub_dir, sample_dir, f"{tag}.df")
    this_split_df = pd.read_hdf(filename, key="split")
    this_n_split = this_split_df.n_split.iloc[0]

    with pd.HDFStore(filename, mode='r') as store:
        keys = store.keys() 
        # print("Keys:", keys)

    for i in range(this_n_split):
        mcnu_key = f"mcnu_{i}"
        this_nudf = pd.read_hdf(filename, key=mcnu_key)

        # ====== add stuff to the df ======
        try:
            # # add multiindex column index "mc" so that branch names match evt_df
            # this_nudf.columns = pd.MultiIndex.from_tuples([tuple(["mc"] + list(c)) for c in this_nudf.columns])     # match # of column levels

            # # event categories
            # this_nudf.loc[:,'topo_categ'] = get_topo_category(this_nudf)
            # this_nudf.loc[:,'genie_categ'] = get_genie_category(this_nudf)

            # # opening angle
            # this_nudf = add_opening_angle(this_nudf, nu=True)

            # tki
            tki_var_names = ["del_alpha", "del_phi", "del_Tp", "del_p", "del_Tp_x", "del_Tp_y"]
            mc_mudf = this_nudf.mc.mu
            mc_pdf = this_nudf.mc.p
            mc_P_mu_col = pad_column_name(("totp",), mc_mudf)
            mc_P_p_col = pad_column_name(("totp",), mc_pdf)
            tki_mc = get_cc1p0pi_tki(mc_mudf, mc_pdf, mc_P_mu_col, mc_P_p_col)
            for var_name in tki_var_names:
                this_nudf = multicol_add(this_nudf, tki_mc[var_name].rename("{}".format(var_name)))

            # slimmed column for G4 syst
            # cols = pd.MultiIndex.from_product(
            #     [["G4"], [f"univ_{i}" for i in range(multisim_nuniv)]],
            # )
            # systs_slim = pd.DataFrame(
            #     1.0,
            #     index=nuidx,
            #     columns=cols,
            # )


        except:
            # raise ValueError(f"Error for {evt_key}")
            print(f"Error for {mcnu_key}")
            continue
        # ==================================

        # overwrite df with the same structure as the original df
        with pd.HDFStore(filename, mode='r+') as store:
            store.put(mcnu_key, this_nudf)

# Modify evtdf & nudf

In [5]:
from makedf.geniesyst import *
from makedf.bnbsyst import *

In [6]:
# genie_tag = "Ar23p"
# syst_type = f"genie-{genie_tag}"
# syst_list = qe_genie_systematics
# df_tag = ""
# subdir = f"genie_wgts-{genie_tag}"

# if genie_tag == "CCQE":
#     df_tag="_geniewgts_CCQE"
#     subdir = "genie_wgts-CCQE"

# # ===== files to process =====
# file_dir = "/exp/sbnd/data/users/munjung/xsec/2025Spring_v10_06_00_09"
# n_max_concat = 3
# mc_keys2load = ['hdr', 'mcnu', 'evt'] 

# concat_dfs = load_and_concat_mc_dfs(
#     file_dir=file_dir,
#     chunk_tags=generate_tags("ak"),
#     df_tag=df_tag,
#     keys2load=mc_keys2load,
#     n_max_concat=n_max_concat,
#     sub_dir="MC",
#     sample_dir="BNB_cosmics/"+subdir
# )

In [7]:
file_dir = "/exp/sbnd/data/users/munjung/xsec/2025Spring_v10_06_00_09"
sub_dir="MC"

# for genie_tag in ["RES", "nonRES", "DIS", "Other", "Ar23p"]:
for genie_tag in ["Ar23p"]:

    sample_dir="BNB_cosmics"

    syst_type = f"genie-{genie_tag}"
    df_tag = ""
    subdir = f"genie_wgts-{genie_tag}"
    split_tags = generate_tags("bl")

    if genie_tag == "CCQE":
        df_tag="_geniewgts_CCQE"
        subdir = "genie_wgts-CCQE"
        split_tags = generate_tags("bl")


    elif genie_tag == "Ar23p":
        df_tag = ""
        subdir = f"genie_wgts-{genie_tag}"
        # split_tags = ["batch1_"+dt for dt in generate_tags("ah")] + ["batch2_"+dt for dt in generate_tags("ao")]

    sample_dir = sample_dir + "/" + subdir

    for tag in split_tags:
        print("file tag:", tag)
        filename = path.join(file_dir, sub_dir, sample_dir, f"{tag}{df_tag}.df")
        this_split_df = pd.read_hdf(filename, key="split")
        this_n_split = this_split_df.n_split.iloc[0]

        with pd.HDFStore(filename, mode='r') as store:
            keys = store.keys() 
            # print("Keys:", keys)

        for i in range(this_n_split):
            mcnu_key = f"mcnu_{i}"
            this_nudf = pd.read_hdf(filename, key=mcnu_key)
            evt_key = f"evt_{i}"
            this_evtdf = pd.read_hdf(filename, key=evt_key)

            # ====== add stuff to the df ======
            # # try:
            #     # nudf column name
            this_nudf.columns = pd.MultiIndex.from_tuples([tuple(["mc"] + list(c)) for c in this_nudf.columns])     # match # of column levels

            # get event categories
            this_evtdf.loc[:,'topo_categ'] = get_topo_category(this_evtdf)
            this_evtdf.loc[:,'genie_categ'] = get_genie_category(this_evtdf)
            this_nudf.loc[:,'topo_categ'] = get_topo_category(this_nudf)
            this_nudf.loc[:,'genie_categ'] = get_genie_category(this_nudf)

            # # opening angle
            this_evtdf = add_opening_angle(this_evtdf)
            this_evtdf = add_opening_angle(this_evtdf, truth=True)
            this_nudf = add_opening_angle(this_nudf, nu=True)

            this_evtdf = add_track_direction(this_evtdf)

            # add tki
            tki_var_names = ["del_alpha", "del_phi", "del_Tp", "del_p", "del_Tp_x", "del_Tp_y"]

            slc_mudf = this_evtdf.mu.pfp.trk.truth.p
            slc_pdf = this_evtdf.p.pfp.trk.truth.p
            slc_P_mu_col = pad_column_name(("totp",), slc_mudf)
            slc_P_p_col = pad_column_name(("totp",), slc_pdf)
            tki_reco = get_cc1p0pi_tki(slc_mudf, slc_pdf, slc_P_mu_col, slc_P_p_col)
            for var_name in tki_var_names:
                this_evtdf = multicol_add(this_evtdf, tki_reco[var_name].rename("mc_" + var_name))

            mc_mudf = this_nudf.mc.mu
            mc_pdf = this_nudf.mc.p
            mc_P_mu_col = pad_column_name(("totp",), mc_mudf)
            mc_P_p_col = pad_column_name(("totp",), mc_pdf)
            tki_mc = get_cc1p0pi_tki(mc_mudf, mc_pdf, mc_P_mu_col, mc_P_p_col)
            for var_name in tki_var_names:
                    this_nudf = multicol_add(this_nudf, tki_mc[var_name].rename("{}".format(var_name)))

            for syst_name in ar23p_genie_systematics:
                type_name = this_nudf[("mc", syst_name)].columns[0][0]
                if "univ" in type_name:
                    print("skipping multisim")
                    continue
                print(type_name)
                this_nudf[("mc", syst_name, "univ_0", "")] = this_nudf[("mc", syst_name, type_name)].copy()
                this_evtdf[("mc", syst_name, "univ_0", "", "", "", "")] = this_evtdf[("mc", syst_name, type_name, "", "", "", "")].copy()

            # except:
            #     # raise ValueError(f"Error for {evt_key}")
            #     print(f"Error for {mcnu_key}")
            #     continue
            # ==================================

            # overwrite df with the same structure as the original df
            with pd.HDFStore(filename, mode='r+') as store:
                store.put(mcnu_key, this_nudf)
                store.put(evt_key, this_evtdf)

file tag: aa
adding track direction


/exp/sbnd/app/users/munjung/xsec/cafpyana_2026Jan17/cafpyana/envs/venv_py310_cafpyana/lib/python3.10/site-packages/pandas/core/arraylike.py:399: RuntimeWarning: invalid value encountered in arccos
  result = getattr(ufunc, method)(*inputs, **kwargs)
/exp/sbnd/app/users/munjung/xsec/cafpyana_2026Jan17/cafpyana/envs/venv_py310_cafpyana/lib/python3.10/site-packages/pandas/core/arraylike.py:399: RuntimeWarning: invalid value encountered in arccos
  result = getattr(ufunc, method)(*inputs, **kwargs)


ps1
ps1
ps1
ps1
ps1
ps1
ps1
ps1
ps1
ps1
ps1
ps1
ps1
ps1
ps1
ps1
ps1
ps1
ps1
ps1
ps1
ps1
ps1
ps1
ps1
ps1
ps1
ps1
ps1
ps1
ps1
ps1
ps1
ps1
ps1
ps1
ps1
ps1
ps1
ps1
ps1
ps1
adding track direction


/exp/sbnd/app/users/munjung/xsec/cafpyana_2026Jan17/cafpyana/envs/venv_py310_cafpyana/lib/python3.10/site-packages/pandas/core/arraylike.py:399: RuntimeWarning: invalid value encountered in arccos
  result = getattr(ufunc, method)(*inputs, **kwargs)
/exp/sbnd/app/users/munjung/xsec/cafpyana_2026Jan17/cafpyana/envs/venv_py310_cafpyana/lib/python3.10/site-packages/pandas/core/arraylike.py:399: RuntimeWarning: invalid value encountered in arccos
  result = getattr(ufunc, method)(*inputs, **kwargs)


ps1
ps1
ps1
ps1
ps1
ps1
ps1
ps1
ps1
ps1
ps1
ps1
ps1
ps1
ps1
ps1
ps1
ps1
ps1
ps1
ps1
ps1
ps1
ps1
ps1
ps1
ps1
ps1
ps1
ps1
ps1
ps1
ps1
ps1
ps1
ps1
ps1
ps1
ps1
ps1
ps1
ps1
adding track direction


...

/exp/sbnd/app/users/munjung/xsec/cafpyana_2026Jan17/cafpyana/envs/venv_py310_cafpyana/lib/python3.10/site-packages/pandas/core/arraylike.py:399: RuntimeWarning: invalid value encountered in arccos
  result = getattr(ufunc, method)(*inputs, **kwargs)
/exp/sbnd/app/users/munjung/xsec/cafpyana_2026Jan17/cafpyana/envs/venv_py310_cafpyana/lib/python3.10/site-packages/pandas/core/arraylike.py:399: RuntimeWarning: invalid value encountered in arccos
  result = getattr(ufunc, method)(*inputs, **kwargs)


ps1
ps1
ps1
ps1
ps1
ps1
ps1
ps1
ps1
ps1
ps1
ps1
ps1
ps1
ps1
ps1
ps1
ps1
ps1
ps1
ps1
ps1
ps1
ps1
ps1
ps1
ps1
ps1
ps1
ps1
ps1
ps1
ps1
ps1
ps1
ps1
ps1
ps1
ps1
ps1
ps1
ps1
adding track direction


/exp/sbnd/app/users/munjung/xsec/cafpyana_2026Jan17/cafpyana/envs/venv_py310_cafpyana/lib/python3.10/site-packages/pandas/core/arraylike.py:399: RuntimeWarning: invalid value encountered in arccos
  result = getattr(ufunc, method)(*inputs, **kwargs)


ps1
ps1
ps1
ps1
ps1
ps1
ps1
ps1
ps1
ps1
ps1
ps1
ps1
ps1
ps1
ps1
ps1
ps1
ps1
ps1
ps1
ps1
ps1
ps1
ps1
ps1
ps1
ps1
ps1
ps1
ps1
ps1
ps1
ps1
ps1
ps1
ps1
ps1
ps1
ps1
ps1
ps1
file tag: ak


FileNotFoundError: File /exp/sbnd/data/users/munjung/xsec/2025Spring_v10_06_00_09/MC/BNB_cosmics/genie_wgts-Ar23p/ak.df does not exist

In [ ]:
this_evtdf.mc

In [ ]:
# check
# with pd.HDFStore(filename, mode='r') as store:
#     keys = store.keys()       # list of all keys in the file
#     print("Keys:", keys)

# type_key = "mcnu"
# for i in range(this_n_split):
#     this_key = f"{type_key}_{i}"
#     this_df = pd.read_hdf(filename, key=this_key)

In [ ]:
this_df.mc

# Match events across variation samples

** must run on EAF **

In [ ]:
sys.path.append('/exp/sbnd/app/users/munjung/xsec/cafpyana_2026Jan17/cafpyana') # absolute path for running on EAF
from analysis_village.numucc_1p0pi.makedf.selections import *

In [ ]:
n_max_concat = 100

mc_file = "/scratch/7DayLifetime/munjung/detvar_test_YZ.df"
mc_split_df = pd.read_hdf(mc_file, key="split")
mc_keys2load = ['hdr', 'evt', 'trk', 'meta', 'mcnu'] 
mc_dfs = load_dfs(mc_file, mc_keys2load, n_max_concat=n_max_concat)
mc_hdr_df_YZ = mc_dfs['meta']
mc_evt_df_YZ = mc_dfs['evt']

mc_file = "/scratch/7DayLifetime/munjung/detvar_test_XThetaXW.df"
mc_split_df = pd.read_hdf(mc_file, key="split")
mc_dfs = load_dfs(mc_file, mc_keys2load, n_max_concat=n_max_concat)
mc_hdr_df_XThetaXW = mc_dfs['meta']
mc_evt_df_XThetaXW = mc_dfs['evt']

In [ ]:
mc_evt_df_YZ.mc.E
mc_evt_df_YZ.reset_index().set_index(["__ntuple", "entry","rec.mc.nu..index",("mc","E")]).index.unique()

In [ ]:
mc_hdr_df_YZ.reset_index().set_index(["__ntuple", "entry","rec.mc.nu..index","E"]).index.unique()

In [ ]:
dfs = [mc_hdr_df_XThetaXW, mc_hdr_df_YZ]

# match common events
for didx, df in enumerate(dfs):
    df = df.reset_index().set_index(["run","subrun","evt","E"])
    print(len(df.index))

# for didx, df in enumerate(dfs):
    idxs = df.index
    if didx == 0:
        common_idxs = idxs
    else:
        common_idxs = common_idxs.intersection(idxs)
print(len(common_idxs))

last_level_values = common_idxs.get_level_values("E")
mask_notnan = ~pd.isna(last_level_values)
common_idx_notnan = common_idxs[mask_notnan]

In [ ]:
common_idx_notnan
mc_evt_df_XThetaXW

In [ ]:
file_dir = "/exp/sbnd/data/users/munjung/xsec/2025Spring_v10_06_00_10"

# variations
this_variation = "WireMod"
syst_keys = ["CV", "WireMod_XThetaXW_updatecalo_ccal_p", "WireMod_YZ_updatecalo_ccal_m"]

In [ ]:
syst_dfs = {}
for sidx, syst_key in enumerate(syst_keys):
    filename = "SystVar_{}_wtrks.df".format(syst_key)
    mc_file = path.join(file_dir, filename)
    mc_split_df = pd.read_hdf(mc_file, key="split")
    mc_n_split = get_n_split(mc_file)
    print("mc_n_split: %d" %(mc_n_split))
    print_keys(mc_file)

    n_max_concat = 100
    mc_keys2load = ['hdr', 'evt', 'trk'] 
    mc_dfs = load_dfs(mc_file, mc_keys2load, n_max_concat=n_max_concat)
    mc_hdr_df = mc_dfs['hdr']
    print("total pot: %.3e" %(mc_hdr_df["pot"].sum()))
    mc_evt_df = mc_dfs['evt']
    mc_trk_df = mc_dfs['trk']
    mc_trk_df = mc_trk_df[mc_trk_df.pfp.trk.producer != 4294967295]
    mask = (mc_trk_df.pfp.trk.len > 0) &\
         (mc_trk_df.pfp.pfochar.vtxdist < 100) #&\
    mc_trk_df = mc_trk_df[mask]

    nlevels = len(mc_evt_df.columns.levels)
    index_names = mc_evt_df.index.names
    mc_hdr_df.columns = pd.MultiIndex.from_tuples([tuple([str(c)] +[""] * (nlevels-1)) for c in mc_hdr_df.columns]) 
    mc_evt_df = multicol_merge(mc_evt_df.reset_index(), 
                               mc_hdr_df.reset_index(),
                               left_on=["__ntuple", "entry"],
                               right_on=["__ntuple", "entry"],
                               how="left"
                               ) 
    mc_evt_df = mc_evt_df.set_index(index_names, verify_integrity=True) 

    # need to match the nu_Es across files
    mc_evt_df["nu_E"] = mc_evt_df.mc.E

    syst_dfs[syst_key] = mc_evt_df
    syst_dfs[syst_key+"_trk"] = mc_trk_df

del mc_hdr_df
del mc_evt_df

In [ ]:
# match common events
for k in syst_keys:
    syst_dfs[k] = syst_dfs[k].reset_index().set_index(["run","subrun","evt","nu_E"])
    print(len(syst_dfs[k].index))

for kidx, k in enumerate(syst_keys):
    idxs = syst_dfs[k].index
    if kidx == 0:
        common_idxs = idxs
    else:
        common_idxs = common_idxs.intersection(idxs)
print(len(common_idxs))

last_level_values = common_idxs.get_level_values("nu_E")
mask_notnan = ~pd.isna(last_level_values)
common_idx_notnan = common_idxs[mask_notnan]

In [ ]:
for k in syst_keys:
    common_nu_df = syst_dfs[k].loc[common_idx_notnan]
    common_nu_df = common_nu_df.reset_index().set_index(["run","subrun","evt","__ntuple"])
    common_nu_idx = common_nu_df.index
    del common_nu_df

    common_nu_idx = common_nu_idx.drop_duplicates()
    common_df = syst_dfs[k].reset_index().set_index(["run","subrun","evt","__ntuple"])
    common_df = common_df.loc[common_nu_idx]

    this_pot = common_df[common_df["first_in_subrun"] == 1]["pot"].sum()
    print(this_pot)
    if k == "CV": 
        cv_pot = this_pot
        common_df["pot_weight"] = np.ones(len(common_df))
    else:
        common_df["pot_weight"] = np.ones(len(common_df)) * cv_pot / this_pot

    syst_dfs[k] = common_df

In [ ]:
for syst_key in syst_keys:
    syst_dfs[syst_key] = syst_dfs[syst_key].reset_index().set_index(list(syst_dfs[f'{syst_key}_trk'].index.names)[:-1])
    syst_dfs[syst_key+"_trk"] = get_valid_trks(syst_dfs[syst_key+"_trk"])
    syst_dfs[syst_key+"_trk"] = match_trkdf_to_slcdf(syst_dfs[syst_key+"_trk"], syst_dfs[syst_key])

In [ ]:
# save matched dfs 
save_filename = path.join(file_dir, "BNB_cosmics-matched-{}.h5".format(this_variation))
with pd.HDFStore(save_filename, "a") as store:
    for syst_key in syst_dfs.keys():
        print(syst_key)
        store.put(syst_key, syst_dfs[syst_key])
        store.put(syst_key+"_trk", syst_dfs[syst_key+"_trk"])

# Add to syst dfs

In [ ]:
import sys
sys.path.append('/exp/sbnd/app/users/munjung/xsec/cafpyana_2026Jan17/cafpyana') # absolute path for running on EAF
from analysis_village.numucc_1p0pi.utils import *
from analysis_village.numucc_1p0pi.categories import *
from pyanalib.variable_calculator import *

In [ ]:
# load matched dfs 
# this_variation = "WireMod"
# syst_keys = ["CV", "WireMod_XThetaXW", "WireMod_YZ"]
this_variation = "SCE"
syst_keys = ["CV", "0xSCE", "2xSCE"]
scratch_file_dir = "/scratch/7DayLifetime/munjung/xsec/detvars"
save_filename = path.join(scratch_file_dir, "BNB_cosmics-matched-{}.h5".format(this_variation))
with pd.HDFStore(save_filename, "r+") as store:
    for syst_key in syst_keys:
        print(syst_key)
        this_evtdf = store.get(syst_key)
        # this_trkdf = store.get(syst_key+"_trk")

        # chimu_avg = avg_chi2(this_trkdf, "chi2_muon")
        # this_trkdf[("pfp", "trk", "chi2pid", "avg", "chi2_muon", "")] = chimu_avg

        # chip_avg = avg_chi2(this_trkdf, "chi2_proton")
        # this_trkdf[("pfp", "trk", "chi2pid", "avg", "chi2_proton", "")] = chip_avg

        # store.put(syst_key+"_trk", this_trkdf)

        this_evtdf.loc[:,'topo_categ'] = get_topo_category(this_evtdf)
        this_evtdf.loc[:,'genie_categ'] = get_genie_category(this_evtdf)
        store.put(syst_key, this_evtdf)


In [ ]:
print(filename)